In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nfl_data_py as nfl
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from scipy import stats
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv('../data/processed/merged_data_with_college.csv')

In [2]:
df_rb = df[df['pos_group'] == 'RB']
for pct in [70, 75, 80, 85]:
    val = df_rb['w_av'].quantile(pct/100)
    print(f"{pct}th percentile: {val}")

70th percentile: 20.0
75th percentile: 24.0
80th percentile: 28.0
85th percentile: 34.0


In [3]:
print(df[(df['w_av'] >= 28) & (df['pos'] == 'RB')][['player_name', 'college', 'pos', 'w_av', 'pick']])

          player_name       college pos  w_av  pick
975   Shaun Alexander       Alabama  RB  68.0    19
983      Thomas Jones      Virginia  RB  62.0     7
985       Jamal Lewis     Tennessee  RB  69.0     5
997      Kevan Barlow    Pittsburgh  RB  36.0    80
998   Michael Bennett     Wisconsin  RB  34.0    27
...               ...           ...  ..   ...   ...
1419   Kenneth Walker  Michigan St.  RB  28.0    41
1422   Kyren Williams    Notre Dame  RB  31.0   164
1424     Devon Achane     Texas A&M  RB  31.0    84
1429     Jahmyr Gibbs       Alabama  RB  40.0    12
1434   Bijan Robinson         Texas  RB  36.0     8

[99 rows x 5 columns]


### Going to go with w_av >= 28 for RBs. Most rbs are replaceable so we'll see what we get

In [4]:
df_rb['is_hit'] = (df_rb['w_av'] >= 28).astype(int)
print(f"Hits: {df_rb['is_hit'].sum():.0f}, Busts: {(df_rb['is_hit'] == 0).sum()}")

Hits: 100, Busts: 400


In [5]:
features = ['ht', 'wt', 'forty', 'vertical', 'broad_jump']
df_rb_clean = df_rb.dropna(subset=features + ['is_hit'])
print(f"RBs with all features: {len(df_rb_clean)}")
X = df_rb_clean[features]
y = df_rb_clean['is_hit']

print(f"Shape: {X.shape}")
print(y.value_counts())

RBs with all features: 387
Shape: (387, 5)
is_hit
0    312
1     75
Name: count, dtype: int64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size= 0.25,
                                                    random_state=42,
                                                    stratify=y)

## standardizing data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## running LogisiticRegression
model = LogisticRegression(random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test,y_pred))

## running RandomForest
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, rf_pred):.3f}")
print(classification_report(y_test,rf_pred))

## running GradientBoost
gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train_scaled, y_train)
gb_pred = gb_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, gb_pred):.3f}")
print(classification_report(y_test, gb_pred))

for name, mod in [('LogReg', LogisticRegression(random_state=42, class_weight='balanced')),
                   ('RF', RandomForestClassifier(random_state=42, class_weight='balanced')),
                   ('GB', GradientBoostingClassifier(random_state=42))]:
    scores = cross_val_score(mod, X_train_scaled, y_train, cv=5, scoring='accuracy')
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")

Accuracy: 0.515
              precision    recall  f1-score   support

           0       0.83      0.50      0.62        78
           1       0.22      0.58      0.32        19

    accuracy                           0.52        97
   macro avg       0.52      0.54      0.47        97
weighted avg       0.71      0.52      0.56        97

Accuracy: 0.784
              precision    recall  f1-score   support

           0       0.80      0.97      0.88        78
           1       0.00      0.00      0.00        19

    accuracy                           0.78        97
   macro avg       0.40      0.49      0.44        97
weighted avg       0.64      0.78      0.71        97

Accuracy: 0.794
              precision    recall  f1-score   support

           0       0.82      0.96      0.88        78
           1       0.40      0.11      0.17        19

    accuracy                           0.79        97
   macro avg       0.61      0.53      0.52        97
weighted avg       0.73   

### same pattern as QB model, RF and GB have high accuracy but barely predict any hits. Logistic regression is finding hits (58% recall) but with low precision.

For example, in GB:
- bust precision is 0.82: when it predicts bust, it is correct 82% of the time
- hit precision is 0.40: when it predicts hit, it is correct 40% of the time
- bust recall is 0.96: so it found 96% of the actual busts
- hit recall is 0.11: so it found 11% of the actual hits

So the accuracy of 79.4 sounds good, but it is misleading because the model is mostly just guessing bust.

For the report the most important number is hit recall, as that's how many successful players our model actual identified. A model that never predicts hit is useless for finding undervalued players, even if its accuracy is high.

In [7]:
college_features = ['career_g', 'career_rush_att', 'career_rush_yds', 'career_rush_ya',
                    'career_rush_td', 'career_rush_yg',
                    'career_rec', 'career_rec_yds', 'career_rec_yr', 'career_rec_td',
                    'last_g', 'last_rush_att', 'last_rush_yds', 'last_rush_ya',
                    'last_rush_td', 'last_rush_yg',
                    'last_rec', 'last_rec_yds', 'last_rec_yr', 'last_rec_td']

all_features = features + college_features
df_rb_college = df_rb.dropna(subset=all_features + ['is_hit'])
print(f"RBs with all features: {len(df_rb_college)}")

RBs with all features: 355


In [8]:
X = df_rb_college[all_features]
y = df_rb_college['is_hit']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size= 0.25,
                                                    random_state=42,
                                                    stratify=y)

## standardizing data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## running LogisiticRegression
model = LogisticRegression(random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test,y_pred))

## running RandomForest
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, rf_pred):.3f}")
print(classification_report(y_test,rf_pred))

## running GradientBoost
gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train_scaled, y_train)
gb_pred = gb_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, gb_pred):.3f}")
print(classification_report(y_test, gb_pred))

for name, mod in [('LogReg', LogisticRegression(random_state=42, class_weight='balanced')),
                   ('RF', RandomForestClassifier(random_state=42, class_weight='balanced')),
                   ('GB', GradientBoostingClassifier(random_state=42))]:
    scores = cross_val_score(mod, X_train_scaled, y_train, cv=5, scoring='accuracy')
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")

Accuracy: 0.663
              precision    recall  f1-score   support

           0       0.85      0.70      0.77        71
           1       0.30      0.50      0.38        18

    accuracy                           0.66        89
   macro avg       0.57      0.60      0.57        89
weighted avg       0.74      0.66      0.69        89

Accuracy: 0.798
              precision    recall  f1-score   support

           0       0.80      0.99      0.89        71
           1       0.50      0.06      0.10        18

    accuracy                           0.80        89
   macro avg       0.65      0.52      0.49        89
weighted avg       0.74      0.80      0.73        89

Accuracy: 0.798
              precision    recall  f1-score   support

           0       0.82      0.96      0.88        71
           1       0.50      0.17      0.25        18

    accuracy                           0.80        89
   macro avg       0.66      0.56      0.57        89
weighted avg       0.75   

In [9]:
## Adjust prediction threshold for GB
probs = gb_model.predict_proba(X_test_scaled)[:, 1]

for thresh in [0.50, 0.40, 0.30, 0.20]:
    pred = (probs >= thresh).astype(int)
    print(f"\nThreshold: {thresh}")
    print(classification_report(y_test, pred))


Threshold: 0.5
              precision    recall  f1-score   support

           0       0.82      0.96      0.88        71
           1       0.50      0.17      0.25        18

    accuracy                           0.80        89
   macro avg       0.66      0.56      0.57        89
weighted avg       0.75      0.80      0.76        89


Threshold: 0.4
              precision    recall  f1-score   support

           0       0.84      0.93      0.88        71
           1       0.50      0.28      0.36        18

    accuracy                           0.80        89
   macro avg       0.67      0.60      0.62        89
weighted avg       0.77      0.80      0.77        89


Threshold: 0.3
              precision    recall  f1-score   support

           0       0.83      0.89      0.86        71
           1       0.38      0.28      0.32        18

    accuracy                           0.76        89
   macro avg       0.61      0.58      0.59        89
weighted avg       0.74   

### After adjusting the threshold the hit recall doubled from 17% to 39% while precision stayed decent at 44%. This means we found 39% of the actual hits and our model gets every hit it predicts right about 44% of the time. Which is good for this, we can cast a wider net and flag more candidates and a GM can always do further evaluation on the flagged players.

In [10]:
features = ['ht', 'wt', 'forty', 'vertical', 'broad_jump']
college_features = ['career_g', 'career_rush_att', 'career_rush_yds', 'career_rush_ya',
                    'career_rush_td', 'career_rush_yg',
                    'career_rec', 'career_rec_yds', 'career_rec_yr', 'career_rec_td',
                    'last_g', 'last_rush_att', 'last_rush_yds', 'last_rush_ya',
                    'last_rush_td', 'last_rush_yg',
                    'last_rec', 'last_rec_yds', 'last_rec_yr', 'last_rec_td']

all_features = features + college_features
df_rb_draft = df_rb.dropna(subset=all_features + ['pick'])
X = df_rb_draft[all_features]
y = df_rb_draft['pick']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}\n")

for name, mod in [('Ridge', Ridge(random_state=42)),
                   ('Lasso', Lasso(random_state=42)),
                   ('RF', RandomForestRegressor(random_state=42)),
                   ('GB', GradientBoostingRegressor(random_state=42))]:
    mod.fit(X_train_scaled, y_train)
    pred = mod.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    print(f"{name}:")
    print(f"  MAE: {mae:.1f} picks off")
    print(f"  RMSE: {rmse:.1f}")
    print(f"  R²: {r2:.3f}")
    scores = cross_val_score(mod, X_train_scaled, y_train, cv=5, scoring='neg_mean_absolute_error')
    print(f"  CV MAE: {-scores.mean():.1f} (+/- {scores.std():.1f})\n")

Train: (266, 25), Test: (89, 25)

Ridge:
  MAE: 50.1 picks off
  RMSE: 59.8
  R²: 0.170
  CV MAE: 49.0 (+/- 6.0)

Lasso:
  MAE: 48.8 picks off
  RMSE: 57.6
  R²: 0.228
  CV MAE: 49.2 (+/- 5.7)

RF:
  MAE: 49.3 picks off
  RMSE: 58.0
  R²: 0.217
  CV MAE: 50.2 (+/- 4.7)

GB:
  MAE: 51.1 picks off
  RMSE: 60.5
  R²: 0.150
  CV MAE: 50.8 (+/- 4.2)



### LASSO is the best draft prediction model, it is off by about 49 picks with the lowest RMSE. It also explains almost 23% of the variance in draft position. Good CV MAE as well.

## Now creating the full model for RB using GB with 0.2 threshold for hit/bust model and using Lasso for the draft model

In [11]:
X_all = df_rb_college[all_features]
y_hit = df_rb_college['is_hit']
y_pick = df_rb_college['pick']

scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)

## Hit probability
gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_all_scaled, y_hit)
hit_prob = gb_model.predict_proba(X_all_scaled)[:, 1]

## Predicted pick
lasso_model = Lasso(random_state=42)
lasso_model.fit(X_all_scaled, y_pick)
pred_pick = lasso_model.predict(X_all_scaled)

## Combine results
results = df_rb_college[['player_name', 'college', 'season', 'pick', 'w_av', 'is_hit']].copy()
results['hit_probability'] = hit_prob.round(3)
results['predicted_pick'] = pred_pick.round(1)
results['actual_pick'] = results['pick']
results['pick_difference'] = results['actual_pick'] - results['predicted_pick']

## Undervalued = high hit probability + drafted later than predicted
print("TOP RB PROSPECTS (highest hit probability): ")
print(results[results['predicted_pick'] > 0].sort_values('hit_probability', ascending=False).head(20)[
    ['player_name', 'college', 'season', 'actual_pick', 'predicted_pick', 'hit_probability', 'w_av']
].to_string(index=False))

TOP RB PROSPECTS (highest hit probability): 
        player_name           college  season  actual_pick  predicted_pick  hit_probability  w_av
       Duke Johnson        Miami (FL)    2015           77            99.7            0.982  28.0
        Kareem Hunt            Toledo    2017           86           105.9            0.975  52.0
      Melvin Gordon         Wisconsin    2015           15            64.0            0.960  51.0
LaDainian Tomlinson               TCU    2001            5            72.6            0.952 129.0
        Breece Hall          Iowa St.    2022           36            58.5            0.946  28.0
     Bijan Robinson             Texas    2023            8            68.7            0.946  36.0
    Jonathan Taylor         Wisconsin    2020           41            25.2            0.935  58.0
      Derrick Henry           Alabama    2016           45            26.8            0.934  85.0
        Aaron Jones     Texas-El Paso    2017          182            89.

In [12]:
## Value score: high hit probability + drafted later than model expected
results['value_score'] = results['hit_probability'] * 100 + results['pick_difference']

print("\nMOST UNDERVALUED RBs (hit probability > 0.2 AND drafted later than predicted):")
undervalued = results[(results['hit_probability'] > 0.2) & (results['pick_difference'] > 0) & (results['predicted_pick'] > 0)]
print(undervalued.sort_values('value_score', ascending=False)[
    ['player_name', 'college', 'season', 'actual_pick', 'predicted_pick', 'hit_probability', 'w_av', 'value_score']
].head(15).to_string(index=False))


MOST UNDERVALUED RBs (hit probability > 0.2 AND drafted later than predicted):
    player_name           college  season  actual_pick  predicted_pick  hit_probability  w_av  value_score
 Ahmad Bradshaw          Marshall    2007          250           141.3            0.830  42.0        191.7
    Aaron Jones     Texas-El Paso    2017          182            89.4            0.927  67.0        185.3
   Chris Carson      Oklahoma St.    2017          249           127.9            0.634  29.0        184.5
 Chester Taylor            Toledo    2002          207           115.2            0.839  41.0        175.7
 Michael Turner Northern Illinois    2004          154            78.8            0.899  55.0        165.1
 Justin Forsett        California    2008          233           179.8            0.842  32.0        137.4
   Charles Clay             Tulsa    2011          174           131.0            0.888  29.0        131.8
   Rudi Johnson            Auburn    2001          100          

## !!!!For the model:

We combined two models to identify undervalued draft picks. The first model (Gradient Boosting Classifier) predicts the probability that a RB will have a successful NFL career based on their combine measurables and college stats. The second model (Lasso regression) predicts where a RB should be drafted based on those same features.

The hit/bust model tells us "how good will this player actually be?" and the draft prediction model tells us "how good do NFL teams think this player will be?" When a player has a high hit probability but a later predicted draft pick, it means the model sees something that NFL teams are undervaluing. This gap is where smart teams can find value.

We created a simple value score that combines both signals: hit probability plus pick difference. A positive pick difference means the player was drafted later than expected, so a high value score means the player is both likely to succeed AND was available later than he should have been.

Unlike QB where we used a 50% hit probability threshold, we lowered the RB threshold to 20%. Our threshold analysis showed that lowering the cutoff nearly tripled hit recall (from 17% to 39%) while keeping precision reasonable at 44%. This wider net makes sense for RBs they're a less expensive draft investment than QBs, so casting a broader search for value picks carries less risk.

Our model identified players like Aaron Jones (pick 182, 93% hit probability), Ahmad Bradshaw (pick 250, 83% hit probability), and Michael Turner (pick 154, 90% hit probability) as undervalued. These were all late-round picks who became Pro Bowl caliber players. A team using this model would have flagged these running backs as high-value targets, acquiring starter-level talent at a fraction of the draft capital typically spent on the position.